In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch.optim as optim

# Configurations
DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"
EPOCHS = 5
BATCH_SIZE = 64
LR = 1e-3
MAX_FEATURES = 1000

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# BoW Classifier Model
class BoWClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# train function
def train_model(model, loader, criterion, optimizer):
    model.train()
    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X.float())
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

# evaluate function
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            out = model(X.float())
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(y.cpu().numpy())
    print(classification_report(trues, preds))
    print(confusion_matrix(trues, preds))

if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    X_train, X_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.3, stratify=labels)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

    vectorizer = CountVectorizer(max_features=MAX_FEATURES)
    X_train_bow = vectorizer.fit_transform(X_train).toarray()
    X_val_bow = vectorizer.transform(X_val).toarray()
    X_test_bow = vectorizer.transform(X_test).toarray()

    train_ds = TextDataset(torch.tensor(X_train_bow), torch.tensor(y_train))
    val_ds = TextDataset(torch.tensor(X_val_bow), torch.tensor(y_val))
    test_ds = TextDataset(torch.tensor(X_test_bow), torch.tensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    model = BoWClassifier(input_dim=MAX_FEATURES)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        train_model(model, train_loader, criterion, optimizer)
        print(f"Epoch {epoch+1} validation:")
        evaluate(model, val_loader)

    print("Final Test Performance:")
    evaluate(model, test_loader)


Epoch 1 validation:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89     19047
           1       0.85      0.90      0.88     22174
           2       0.79      0.74      0.76      5429
           3       0.83      0.82      0.83      9004
           4       0.86      0.75      0.80      7513
           5       0.68      0.82      0.75      2354

    accuracy                           0.85     65521
   macro avg       0.82      0.82      0.82     65521
weighted avg       0.85      0.85      0.85     65521

[[16845  1021   104   654   339    84]
 [  569 20016   877   288   210   214]
 [  124  1132  4016   110    34    13]
 [  640   650    47  7423   220    24]
 [  470   427    29   424  5610   553]
 [   56   193    11    27   137  1930]]
Epoch 2 validation:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89     19047
           1       0.85      0.91      0.88     22174
           2       